<a href="https://colab.research.google.com/github/jahid-dev/-Thesis-Data-Driven-Student-Behavior-Modeling-and-Dropout-Prediction-Using-Advanced-ML-Techniques/blob/main/03_UPV_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#SECTION 1: Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import re

from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


#SECTION 2: Google Drive & Project Paths

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Main project directory
BASE_DIR = Path(
    "/content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final"
)

# Dataset directory
DATA_DIR = BASE_DIR / "Datasets"
UPV_DIR = DATA_DIR / "UPV"

# Notebook directory
NOTEBOOK_DIR = BASE_DIR / "Notebooks"

# Results directory
RESULT_DIR = BASE_DIR / "Results"

# Previous audit results
AUDIT_RESULT_DIR = RESULT_DIR / "UPV" / "Audit"

# Previous preprocessing results
PREPROCESSING_RESULT_DIR = RESULT_DIR / "UPV" / "Preprocessing"

# Intermediate datasets produced by 02_UPV_Preprocessing
INTERMEDIATE_DIR = PREPROCESSING_RESULT_DIR / "Intermediate"

# Output directory for this notebook
FEATURE_RESULT_DIR = RESULT_DIR / "UPV" / "Feature_Engineering"

# Create feature engineering output directory
FEATURE_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("BASE_DIR:", BASE_DIR)
print("UPV_DIR:", UPV_DIR)
print("AUDIT_RESULT_DIR:", AUDIT_RESULT_DIR)
print("PREPROCESSING_RESULT_DIR:", PREPROCESSING_RESULT_DIR)
print("INTERMEDIATE_DIR:", INTERMEDIATE_DIR)
print("FEATURE_RESULT_DIR:", FEATURE_RESULT_DIR)

print("\n" + "=" * 70)
print("PATH VERIFICATION")
print("=" * 70)

print("BASE exists:", BASE_DIR.exists())
print("UPV exists:", UPV_DIR.exists())
print("Audit results exist:", AUDIT_RESULT_DIR.exists())
print("Preprocessing results exist:", PREPROCESSING_RESULT_DIR.exists())
print("Intermediate data exists:", INTERMEDIATE_DIR.exists())
print("Feature Engineering directory exists:", FEATURE_RESULT_DIR.exists())

Mounted at /content/drive
PROJECT PATHS
BASE_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final
UPV_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Datasets/UPV
AUDIT_RESULT_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Results/UPV/Audit
PREPROCESSING_RESULT_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Results/UPV/Preprocessing
INTERMEDIATE_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Results/UPV/Preprocessing/Intermediate
FEATURE_RESULT_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Results/UPV/Feature_Engineering

PATH VERIFICATION
BASE exists: True
UPV exists: True
Audit results exist: True
Preprocessing results exist: True
Intermediate data exists: True
Feature Engineering directory exists: True


#SECTION 3: Load Preprocessing Outputs

In [ ]:
print("=" * 70)
print("LOADING PREPROCESSING OUTPUTS")
print("=" * 70)

# Expected intermediate files generated by 02_UPV_Preprocessing
file_2018 = INTERMEDIATE_DIR / "upv_2018_preprocessing_stage.pkl"
file_2021 = INTERMEDIATE_DIR / "upv_2021_preprocessing_stage.pkl"
file_2022 = INTERMEDIATE_DIR / "upv_2022_preprocessing_stage.pkl"


# Verify that all required files exist
required_files = {
    "2018": file_2018,
    "2021": file_2021,
    "2022": file_2022
}

for year, file_path in required_files.items():
    print(f"{year}: {file_path.exists()} -> {file_path.name}")


# Stop execution if any required file is missing
missing_files = [
    year for year, file_path in required_files.items()
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing preprocessing output(s): {', '.join(missing_files)}"
    )


# Load preprocessing-stage datasets
data_2018 = pd.read_pickle(file_2018)
data_2021 = pd.read_pickle(file_2021)
data_2022 = pd.read_pickle(file_2022)


print("\nPreprocessing outputs loaded successfully.")

print("\nDataset shapes:")
print("2018:", data_2018.shape)
print("2021:", data_2021.shape)
print("2022:", data_2022.shape)

LOADING PREPROCESSING OUTPUTS
2018: True -> upv_2018_preprocessing_stage.pkl
2021: True -> upv_2021_preprocessing_stage.pkl
2022: True -> upv_2022_preprocessing_stage.pkl

Preprocessing outputs loaded successfully.

Dataset shapes:
2018: (152446, 178)
2021: (153120, 178)
2022: (159173, 169)


#SECTION 4: Verify Intermediate Datasets

In [ ]:
print("=" * 70)
print("INTERMEDIATE DATASET VERIFICATION")
print("=" * 70)


datasets = {
    "2018": data_2018,
    "2021": data_2021,
    "2022": data_2022
}


for year, df in datasets.items():

    print(f"\n{'-' * 70}")
    print(f"DATASET: {year}")
    print(f"{'-' * 70}")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Unique students:", df["dni_hash"].nunique())

    print(
        "Target column exists:",
        "abandono_hash" in df.columns
    )

    print(
        "Target missing:",
        df["abandono_hash"].isna().sum()
        if "abandono_hash" in df.columns
        else "N/A"
    )

    print(
        "Student identifier missing:",
        df["dni_hash"].isna().sum()
        if "dni_hash" in df.columns
        else "N/A"
    )


# Required columns for feature engineering
required_columns = [
    "dni_hash",
    "abandono_hash",
    "anyo_ingreso",
    "anyo_inicio_estudios",
    "fecha_datos",
    "curso_mas_bajo",
    "curso_mas_alto",
    "baja_fecha"
]


print("\n" + "=" * 70)
print("REQUIRED COLUMN CHECK")
print("=" * 70)


for year, df in datasets.items():

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        print(f"{year}: Missing -> {missing_columns}")
    else:
        print(f"{year}: All required columns available")


print("\nIntermediate dataset verification completed.")

INTERMEDIATE DATASET VERIFICATION

----------------------------------------------------------------------
DATASET: 2018
----------------------------------------------------------------------
Rows: 152446
Columns: 178
Unique students: 20233
Target column exists: True
Target missing: 0
Student identifier missing: 0

----------------------------------------------------------------------
DATASET: 2021
----------------------------------------------------------------------
Rows: 153120
Columns: 178
Unique students: 20264
Target column exists: True
Target missing: 0
Student identifier missing: 0

----------------------------------------------------------------------
DATASET: 2022
----------------------------------------------------------------------
Rows: 159173
Columns: 169
Unique students: 20427
Target column exists: True
Target missing: 0
Student identifier missing: 0

REQUIRED COLUMN CHECK
2018: All required columns available
2021: All required columns available
2022: All required columns

# SECTION 5: Identify Temporal Windows

In [ ]:
print("=" * 70)
print("TEMPORAL FEATURE WINDOW ANALYSIS")
print("=" * 70)


# Temporal feature prefixes identified during the preprocessing audit
TEMPORAL_PREFIXES = [
    "pft_events_",
    "pft_visits_",
    "pft_days_logged_",
    "pft_assignment_submissions_",
    "pft_test_submissions_",
    "pft_total_minutes_",
    "resource_events_",
    "n_resource_days_",
    "n_wifi_days_"
]


def identify_temporal_features(df):
    """
    Identify columns representing temporal or digital behaviour
    using the temporal feature prefixes established during auditing.
    """

    temporal_columns = []

    for column in df.columns:

        if any(
            column.startswith(prefix)
            for prefix in TEMPORAL_PREFIXES
        ):
            temporal_columns.append(column)

    return temporal_columns


# Identify temporal features for each dataset
temporal_features = {}

for year, df in datasets.items():

    temporal_features[year] = identify_temporal_features(df)

    print(
        f"{year}: "
        f"{len(temporal_features[year])} temporal/digital features"
    )


# Display sample temporal columns
print("\n" + "=" * 70)
print("SAMPLE TEMPORAL FEATURES")
print("=" * 70)

for year, columns in temporal_features.items():

    print(f"\n{year}")

    for column in columns[:20]:
        print(" -", column)


# Compare temporal feature counts
print("\n" + "=" * 70)
print("TEMPORAL FEATURE COUNT COMPARISON")
print("=" * 70)

temporal_count_summary = pd.DataFrame({
    "Dataset": list(temporal_features.keys()),
    "Temporal_Feature_Count": [
        len(columns)
        for columns in temporal_features.values()
    ]
})

display(temporal_count_summary)

TEMPORAL FEATURE WINDOW ANALYSIS
2018: 108 temporal/digital features
2021: 108 temporal/digital features
2022: 99 temporal/digital features

SAMPLE TEMPORAL FEATURES

2018
 - pft_events_2018_9
 - pft_days_logged_2018_9
 - pft_visits_2018_9
 - pft_assignment_submissions_2018_9
 - pft_test_submissions_2018_9
 - pft_total_minutes_2018_9
 - n_wifi_days_2018_9
 - resource_events_2018_9
 - n_resource_days_2018_9
 - pft_events_2018_10
 - pft_days_logged_2018_10
 - pft_visits_2018_10
 - pft_assignment_submissions_2018_10
 - pft_test_submissions_2018_10
 - pft_total_minutes_2018_10
 - n_wifi_days_2018_10
 - resource_events_2018_10
 - n_resource_days_2018_10
 - pft_events_2018_11
 - pft_days_logged_2018_11

2021
 - pft_events_2021_9
 - pft_days_logged_2021_9
 - pft_visits_2021_9
 - pft_assignment_submissions_2021_9
 - pft_test_submissions_2021_9
 - pft_total_minutes_2021_9
 - n_wifi_days_2021_9
 - resource_events_2021_9
 - n_resource_days_2021_9
 - pft_events_2021_10
 - pft_days_logged_2021_10
 

,Dataset,Temporal_Feature_Count
0,2018,108
1,2021,108
2,2022,99


# SECTION 6: Parse Temporal Columns

In [ ]:
print("=" * 70)
print("PARSING TEMPORAL COLUMN STRUCTURE")
print("=" * 70)


def parse_temporal_column(column):
    """
    Extract feature family and year/month information
    from temporal digital-behaviour column names.
    """

    for prefix in TEMPORAL_PREFIXES:

        if column.startswith(prefix):

            temporal_part = column[len(prefix):]

            # Match patterns such as 2023_7
            match = re.match(
                r"^(\d{4})_(\d{1,2})$",
                temporal_part
            )

            if match:

                year = int(match.group(1))
                month = int(match.group(2))

                return {
                    "Column": column,
                    "Feature_Family": prefix.rstrip("_"),
                    "Temporal_Year": year,
                    "Temporal_Month": month,
                    "Temporal_Period": f"{year}-{month:02d}"
                }

    return None


# Build temporal inventory
temporal_inventory_rows = []

for dataset_year, columns in temporal_features.items():

    for column in columns:

        parsed = parse_temporal_column(column)

        if parsed is not None:

            parsed["Dataset"] = dataset_year

            temporal_inventory_rows.append(parsed)


temporal_inventory = pd.DataFrame(
    temporal_inventory_rows
)


# Reorder columns
if not temporal_inventory.empty:

    temporal_inventory = temporal_inventory[
        [
            "Dataset",
            "Column",
            "Feature_Family",
            "Temporal_Year",
            "Temporal_Month",
            "Temporal_Period"
        ]
    ]


print(
    "Temporal columns successfully parsed:",
    len(temporal_inventory)
)


print("\n" + "=" * 70)
print("TEMPORAL INVENTORY SAMPLE")
print("=" * 70)

display(
    temporal_inventory.head(20)
)

PARSING TEMPORAL COLUMN STRUCTURE
Temporal columns successfully parsed: 315

TEMPORAL INVENTORY SAMPLE


,Dataset,Column,Feature_Family,Temporal_Year,Temporal_Month,Temporal_Period
0,2018,pft_events_2018_9,pft_events,2018,9,2018-09
1,2018,pft_days_logged_2018_9,pft_days_logged,2018,9,2018-09
2,2018,pft_visits_2018_9,pft_visits,2018,9,2018-09
3,2018,pft_assignment_submissions_2018_9,pft_assignment_submissions,2018,9,2018-09
4,2018,pft_test_submissions_2018_9,pft_test_submissions,2018,9,2018-09
5,2018,pft_total_minutes_2018_9,pft_total_minutes,2018,9,2018-09
6,2018,n_wifi_days_2018_9,n_wifi_days,2018,9,2018-09
7,2018,resource_events_2018_9,resource_events,2018,9,2018-09
8,2018,n_resource_days_2018_9,n_resource_days,2018,9,2018-09
9,2018,pft_events_2018_10,pft_events,2018,10,2018-10


In [ ]:
# Validate parsed temporal periods

print("\n" + "=" * 70)
print("TEMPORAL PERIOD SUMMARY")
print("=" * 70)


period_summary = (
    temporal_inventory
    .groupby("Dataset")
    .agg(
        Number_of_Temporal_Columns=("Column", "count"),
        Number_of_Periods=("Temporal_Period", "nunique"),
        Earliest_Period=("Temporal_Period", "min"),
        Latest_Period=("Temporal_Period", "max")
    )
    .reset_index()
)


display(period_summary)


TEMPORAL PERIOD SUMMARY


,Dataset,Number_of_Temporal_Columns,Number_of_Periods,Earliest_Period,Latest_Period
0,2018,108,12,2018-09,2019-08
1,2021,108,12,2021-09,2022-08
2,2022,99,11,2022-09,2023-07


# SECTION 7: Define Prediction Unit

In [ ]:
print("=" * 70)
print("PREDICTION UNIT DEFINITION")
print("=" * 70)

# The anonymized student identifier represents the longitudinal
# student entity across repeated records.

STUDENT_ID = "dni_hash"
TARGET_COLUMN = "abandono_hash"


# Verify identifier and target
for year, df in datasets.items():

    if STUDENT_ID not in df.columns:
        raise KeyError(
            f"{STUDENT_ID} not found in {year} dataset."
        )

    if TARGET_COLUMN not in df.columns:
        raise KeyError(
            f"{TARGET_COLUMN} not found in {year} dataset."
        )


print("Prediction unit: Student")
print("Student identifier:", STUDENT_ID)
print("Target variable:", TARGET_COLUMN)


# Check number of records and unique students
prediction_unit_summary = []

for year, df in datasets.items():

    prediction_unit_summary.append({
        "Dataset": year,
        "Total_Records": len(df),
        "Unique_Students": df[STUDENT_ID].nunique(),
        "Records_Per_Student":
            round(
                len(df) / df[STUDENT_ID].nunique(),
                2
            )
    })


prediction_unit_summary = pd.DataFrame(
    prediction_unit_summary
)

display(prediction_unit_summary)


print("\nPrediction unit successfully defined.")

PREDICTION UNIT DEFINITION
Prediction unit: Student
Student identifier: dni_hash
Target variable: abandono_hash


,Dataset,Total_Records,Unique_Students,Records_Per_Student
0,2018,152446,20233,7.53
1,2021,153120,20264,7.56
2,2022,159173,20427,7.79



Prediction unit successfully defined.


#SECTION 8: Define Prediction Time

In [ ]:
print("=" * 70)
print("PREDICTION TIME DEFINITION")
print("=" * 70)


# The prediction time is based on the final available
# behavioural observation period in each dataset.

prediction_time_summary = (
    temporal_inventory
    .groupby("Dataset")
    .agg(
        First_Observation_Period=("Temporal_Period", "min"),
        Last_Observation_Period=("Temporal_Period", "max"),
        Number_of_Observation_Periods=("Temporal_Period", "nunique")
    )
    .reset_index()
)


display(prediction_time_summary)


print("\nPrediction-time interpretation:")
print(
    "Student behaviour is represented using information "
    "available within the observed temporal window."
)

print(
    "Withdrawal date is not used as a predictive feature."
)

PREDICTION TIME DEFINITION


,Dataset,First_Observation_Period,Last_Observation_Period,Number_of_Observation_Periods
0,2018,2018-09,2019-08,12
1,2021,2021-09,2022-08,12
2,2022,2022-09,2023-07,11



Prediction-time interpretation:
Student behaviour is represented using information available within the observed temporal window.
Withdrawal date is not used as a predictive feature.


# SECTION 9: Temporal Feature Family Analysis

In [ ]:
print("=" * 70)
print("TEMPORAL FEATURE FAMILY ANALYSIS")
print("=" * 70)


temporal_family_summary = (
    temporal_inventory
    .groupby(
        ["Dataset", "Feature_Family"]
    )
    .agg(
        Number_of_Features=("Column", "count"),
        Number_of_Periods=("Temporal_Period", "nunique"),
        First_Period=("Temporal_Period", "min"),
        Last_Period=("Temporal_Period", "max")
    )
    .reset_index()
)


display(temporal_family_summary)

TEMPORAL FEATURE FAMILY ANALYSIS


,Dataset,Feature_Family,Number_of_Features,Number_of_Periods,First_Period,Last_Period
0,2018,n_resource_days,12,12,2018-09,2019-08
1,2018,n_wifi_days,12,12,2018-09,2019-08
2,2018,pft_assignment_submissions,12,12,2018-09,2019-08
3,2018,pft_days_logged,12,12,2018-09,2019-08
4,2018,pft_events,12,12,2018-09,2019-08
5,2018,pft_test_submissions,12,12,2018-09,2019-08
6,2018,pft_total_minutes,12,12,2018-09,2019-08
7,2018,pft_visits,12,12,2018-09,2019-08
8,2018,resource_events,12,12,2018-09,2019-08
9,2021,n_resource_days,12,12,2021-09,2022-08


In [ ]:
# Temporal feature family coverage by period

print("=" * 70)
print("TEMPORAL FAMILY COVERAGE")
print("=" * 70)


temporal_family_coverage = (
    temporal_inventory
    .groupby(
        ["Dataset", "Feature_Family"]
    )["Temporal_Period"]
    .nunique()
    .reset_index(
        name="Available_Periods"
    )
)


display(temporal_family_coverage)

TEMPORAL FAMILY COVERAGE


,Dataset,Feature_Family,Available_Periods
0,2018,n_resource_days,12
1,2018,n_wifi_days,12
2,2018,pft_assignment_submissions,12
3,2018,pft_days_logged,12
4,2018,pft_events,12
5,2018,pft_test_submissions,12
6,2018,pft_total_minutes,12
7,2018,pft_visits,12
8,2018,resource_events,12
9,2021,n_resource_days,12


In [ ]:
# Validate expected temporal structure

print("=" * 70)
print("TEMPORAL STRUCTURE VALIDATION")
print("=" * 70)


for dataset_year in temporal_inventory["Dataset"].unique():

    subset = temporal_inventory[
        temporal_inventory["Dataset"] == dataset_year
    ]

    print(f"\nDataset {dataset_year}")

    for family in sorted(
        subset["Feature_Family"].unique()
    ):

        family_subset = subset[
            subset["Feature_Family"] == family
        ]

        periods = sorted(
            family_subset["Temporal_Period"].unique()
        )

        print(
            f"{family}: "
            f"{len(periods)} periods "
            f"({periods[0]} → {periods[-1]})"
        )

TEMPORAL STRUCTURE VALIDATION

Dataset 2018
n_resource_days: 12 periods (2018-09 → 2019-08)
n_wifi_days: 12 periods (2018-09 → 2019-08)
pft_assignment_submissions: 12 periods (2018-09 → 2019-08)
pft_days_logged: 12 periods (2018-09 → 2019-08)
pft_events: 12 periods (2018-09 → 2019-08)
pft_test_submissions: 12 periods (2018-09 → 2019-08)
pft_total_minutes: 12 periods (2018-09 → 2019-08)
pft_visits: 12 periods (2018-09 → 2019-08)
resource_events: 12 periods (2018-09 → 2019-08)

Dataset 2021
n_resource_days: 12 periods (2021-09 → 2022-08)
n_wifi_days: 12 periods (2021-09 → 2022-08)
pft_assignment_submissions: 12 periods (2021-09 → 2022-08)
pft_days_logged: 12 periods (2021-09 → 2022-08)
pft_events: 12 periods (2021-09 → 2022-08)
pft_test_submissions: 12 periods (2021-09 → 2022-08)
pft_total_minutes: 12 periods (2021-09 → 2022-08)
pft_visits: 12 periods (2021-09 → 2022-08)
resource_events: 12 periods (2021-09 → 2022-08)

Dataset 2022
n_resource_days: 11 periods (2022-09 → 2023-07)
n_wifi_d

#  SECTION 10: TEMPORAL STRUCTURE VALIDATION

In [ ]:
print("=" * 70)
print("TEMPORAL STRUCTURE VALIDATION")
print("=" * 70)


def validate_temporal_structure(df, dataset_name):
    """
    Validate temporal feature families and their observation periods.
    """

    temporal_columns = [
        col for col in df.columns
        if (
            col.startswith("pft_events_")
            or col.startswith("pft_visits_")
            or col.startswith("pft_days_logged_")
            or col.startswith("pft_assignment_submissions_")
            or col.startswith("pft_test_submissions_")
            or col.startswith("pft_total_minutes_")
            or col.startswith("resource_events_")
            or col.startswith("n_resource_days_")
            or col.startswith("n_wifi_days_")
        )
    ]

    print(f"\nDataset: {dataset_name}")
    print(f"Temporal columns: {len(temporal_columns)}")

    family_periods = {}

    for column in temporal_columns:

        parts = column.split("_")

        if len(parts) < 3:
            continue

        # Find YYYY-MM style temporal suffix
        match = re.search(r"(\d{4})_(\d{1,2})$", column)

        if match:
            year = int(match.group(1))
            month = int(match.group(2))

            period = f"{year}-{month:02d}"

            prefix = column[:match.start()].rstrip("_")

            if prefix not in family_periods:
                family_periods[prefix] = []

            family_periods[prefix].append(period)

    for family, periods in sorted(family_periods.items()):

        periods = sorted(set(periods))

        print(
            f"{family}: "
            f"{len(periods)} periods "
            f"({periods[0]} → {periods[-1]})"
        )

    return family_periods


temporal_structure_2018 = validate_temporal_structure(
    data_2018, "2018"
)

temporal_structure_2021 = validate_temporal_structure(
    data_2021, "2021"
)

temporal_structure_2022 = validate_temporal_structure(
    data_2022, "2022"
)

TEMPORAL STRUCTURE VALIDATION

Dataset: 2018
Temporal columns: 108
n_resource_days: 12 periods (2018-09 → 2019-08)
n_wifi_days: 12 periods (2018-09 → 2019-08)
pft_assignment_submissions: 12 periods (2018-09 → 2019-08)
pft_days_logged: 12 periods (2018-09 → 2019-08)
pft_events: 12 periods (2018-09 → 2019-08)
pft_test_submissions: 12 periods (2018-09 → 2019-08)
pft_total_minutes: 12 periods (2018-09 → 2019-08)
pft_visits: 12 periods (2018-09 → 2019-08)
resource_events: 12 periods (2018-09 → 2019-08)

Dataset: 2021
Temporal columns: 108
n_resource_days: 12 periods (2021-09 → 2022-08)
n_wifi_days: 12 periods (2021-09 → 2022-08)
pft_assignment_submissions: 12 periods (2021-09 → 2022-08)
pft_days_logged: 12 periods (2021-09 → 2022-08)
pft_events: 12 periods (2021-09 → 2022-08)
pft_test_submissions: 12 periods (2021-09 → 2022-08)
pft_total_minutes: 12 periods (2021-09 → 2022-08)
pft_visits: 12 periods (2021-09 → 2022-08)
resource_events: 12 periods (2021-09 → 2022-08)

Dataset: 2022
Temporal 

# SECTION 11: TEMPORAL AGGREGATION STRATEGY

In [ ]:
print("=" * 70)
print("TEMPORAL AGGREGATION STRATEGY")
print("=" * 70)


TEMPORAL_FAMILIES = [
    "pft_events",
    "pft_visits",
    "pft_days_logged",
    "pft_assignment_submissions",
    "pft_test_submissions",
    "pft_total_minutes",
    "resource_events",
    "n_resource_days",
    "n_wifi_days"
]


AGGREGATION_STRATEGY = {
    "total": [
        "pft_events",
        "pft_visits",
        "pft_assignment_submissions",
        "pft_test_submissions",
        "pft_total_minutes",
        "resource_events"
    ],

    "mean": [
        "pft_events",
        "pft_visits",
        "pft_days_logged",
        "pft_assignment_submissions",
        "pft_test_submissions",
        "pft_total_minutes",
        "resource_events",
        "n_resource_days",
        "n_wifi_days"
    ],

    "max": [
        "pft_events",
        "pft_visits",
        "pft_days_logged",
        "pft_assignment_submissions",
        "pft_test_submissions",
        "pft_total_minutes",
        "resource_events",
        "n_resource_days",
        "n_wifi_days"
    ],

    "std": [
        "pft_events",
        "pft_visits",
        "pft_days_logged",
        "pft_assignment_submissions",
        "pft_test_submissions",
        "pft_total_minutes",
        "resource_events",
        "n_resource_days",
        "n_wifi_days"
    ]
}


print("Temporal feature families:")
for family in TEMPORAL_FAMILIES:
    print(f" - {family}")

print("\nAggregation statistics:")
for statistic, families in AGGREGATION_STRATEGY.items():
    print(f" - {statistic}: {len(families)} feature families")

print("\nAggregation strategy defined.")

TEMPORAL AGGREGATION STRATEGY
Temporal feature families:
 - pft_events
 - pft_visits
 - pft_days_logged
 - pft_assignment_submissions
 - pft_test_submissions
 - pft_total_minutes
 - resource_events
 - n_resource_days
 - n_wifi_days

Aggregation statistics:
 - total: 6 feature families
 - mean: 9 feature families
 - max: 9 feature families
 - std: 9 feature families

Aggregation strategy defined.


# SECTION 12: CREATE TEMPORAL FEATURE MAP

In [ ]:
print("=" * 70)
print("TEMPORAL FEATURE MAP")
print("=" * 70)


def create_temporal_feature_map(df, dataset_name):

    rows = []

    for column in df.columns:

        matched_family = None
        year = None
        month = None

        for family in TEMPORAL_FAMILIES:

            prefix = family + "_"

            if column.startswith(prefix):

                matched = re.search(
                    r"_(\d{4})_(\d{1,2})$",
                    column
                )

                if matched:

                    matched_family = family
                    year = int(matched.group(1))
                    month = int(matched.group(2))

                break

        if matched_family is not None:

            rows.append({
                "Dataset": dataset_name,
                "Column": column,
                "Feature_Family": matched_family,
                "Year": year,
                "Month": month,
                "Period": f"{year}-{month:02d}"
            })

    return pd.DataFrame(rows)


temporal_map_2018 = create_temporal_feature_map(
    data_2018, "2018"
)

temporal_map_2021 = create_temporal_feature_map(
    data_2021, "2021"
)

temporal_map_2022 = create_temporal_feature_map(
    data_2022, "2022"
)


temporal_feature_map = pd.concat(
    [
        temporal_map_2018,
        temporal_map_2021,
        temporal_map_2022
    ],
    ignore_index=True
)


display(temporal_feature_map.head(20))

print(
    "\nTotal temporal features mapped:",
    len(temporal_feature_map)
)

TEMPORAL FEATURE MAP


,Dataset,Column,Feature_Family,Year,Month,Period
0,2018,pft_events_2018_9,pft_events,2018,9,2018-09
1,2018,pft_days_logged_2018_9,pft_days_logged,2018,9,2018-09
2,2018,pft_visits_2018_9,pft_visits,2018,9,2018-09
3,2018,pft_assignment_submissions_2018_9,pft_assignment_submissions,2018,9,2018-09
4,2018,pft_test_submissions_2018_9,pft_test_submissions,2018,9,2018-09
5,2018,pft_total_minutes_2018_9,pft_total_minutes,2018,9,2018-09
6,2018,n_wifi_days_2018_9,n_wifi_days,2018,9,2018-09
7,2018,resource_events_2018_9,resource_events,2018,9,2018-09
8,2018,n_resource_days_2018_9,n_resource_days,2018,9,2018-09
9,2018,pft_events_2018_10,pft_events,2018,10,2018-10



Total temporal features mapped: 315


# SECTION 13: MONTHLY → STUDENT-LEVEL AGGREGATION

In [ ]:
def aggregate_temporal_features(df, dataset_name):

    temporal_data = pd.DataFrame({
        "dni_hash": df["dni_hash"],
        "abandono_hash": df["abandono_hash"]
    })

    # Keep one target per student
    student_target = (
        temporal_data[
            ["dni_hash", "abandono_hash"]
        ]
        .drop_duplicates(subset=["dni_hash"])
    )

    result = student_target.copy()

    for family in TEMPORAL_FAMILIES:

        family_columns = [
            col for col in df.columns
            if col.startswith(family + "_")
        ]

        if not family_columns:
            continue

        temp = df[
            ["dni_hash"] + family_columns
        ].copy()

        for column in family_columns:
            temp[column] = pd.to_numeric(
                temp[column],
                errors="coerce"
            )

        grouped = temp.groupby(
            "dni_hash",
            sort=False
        )[family_columns]

        aggregated = pd.DataFrame(index=grouped.mean().index)

        aggregated[f"{family}_total"] = grouped.sum().sum(axis=1)
        aggregated[f"{family}_mean"] = grouped.mean().mean(axis=1)
        aggregated[f"{family}_max"] = grouped.max().max(axis=1)
        aggregated[f"{family}_std"] = grouped.std().mean(axis=1)

        aggregated = aggregated.reset_index()

        result = result.merge(
            aggregated,
            on="dni_hash",
            how="left"
        )

    return result


student_temporal_2018 = aggregate_temporal_features(
    data_2018, "2018"
)

student_temporal_2021 = aggregate_temporal_features(
    data_2021, "2021"
)

student_temporal_2022 = aggregate_temporal_features(
    data_2022, "2022"
)


print("Student-level temporal aggregation completed.")

print(
    "2018:",
    student_temporal_2018.shape
)

print(
    "2021:",
    student_temporal_2021.shape
)

print(
    "2022:",
    student_temporal_2022.shape
)

Student-level temporal aggregation completed.
2018: (20233, 38)
2021: (20264, 38)
2022: (20427, 38)


# SECTION 14: Digital Engagement Features

In [ ]:
print("=" * 70)
print("DIGITAL ENGAGEMENT FEATURES")
print("=" * 70)


def create_engagement_features(df, dataset_name):

    result = df[
        ["dni_hash", "abandono_hash"]
    ].drop_duplicates(
        subset=["dni_hash"]
    ).copy()

    family_pairs = [
        ("pft_events", "events"),
        ("pft_visits", "visits"),
        ("pft_days_logged", "logged_days"),
        ("pft_assignment_submissions", "assignment_submissions"),
        ("pft_test_submissions", "test_submissions"),
        ("pft_total_minutes", "total_minutes"),
        ("resource_events", "resource_events"),
        ("n_resource_days", "resource_days"),
        ("n_wifi_days", "wifi_days")
    ]

    for family, short_name in family_pairs:

        columns = [
            col for col in df.columns
            if col.startswith(family + "_")
        ]

        if not columns:
            continue

        temp = df[
            ["dni_hash"] + columns
        ].copy()

        for col in columns:
            temp[col] = pd.to_numeric(
                temp[col],
                errors="coerce"
            )

        monthly_total = (
            temp.groupby("dni_hash")[columns]
            .sum()
            .sum(axis=1)
        )

        result = result.merge(
            monthly_total.rename(
                f"{short_name}_overall"
            ),
            on="dni_hash",
            how="left"
        )

    return result


engagement_2018 = create_engagement_features(
    data_2018, "2018"
)

engagement_2021 = create_engagement_features(
    data_2021, "2021"
)

engagement_2022 = create_engagement_features(
    data_2022, "2022"
)


print("Digital engagement features created.")

DIGITAL ENGAGEMENT FEATURES
Digital engagement features created.


# SECTION 15: Activity Trend Features

In [ ]:
print("=" * 70)
print("ACTIVITY TREND FEATURES")
print("=" * 70)


TREND_FAMILIES = [
    "pft_events",
    "pft_visits",
    "pft_days_logged",
    "pft_total_minutes",
    "resource_events",
    "n_resource_days",
    "n_wifi_days"
]


def create_trend_features(df):

    result = df[
        ["dni_hash"]
    ].drop_duplicates(
        subset=["dni_hash"]
    ).copy()

    for family in TREND_FAMILIES:

        columns = [
            col for col in df.columns
            if col.startswith(family + "_")
        ]

        if len(columns) < 2:
            continue

        # Preserve chronological order
        periods = []

        for col in columns:

            match = re.search(
                r"_(\d{4})_(\d{1,2})$",
                col
            )

            if match:

                year = int(match.group(1))
                month = int(match.group(2))

                periods.append(
                    (year, month, col)
                )

        periods = sorted(periods)

        ordered_columns = [
            item[2]
            for item in periods
        ]

        temp = df[
            ["dni_hash"] + ordered_columns
        ].copy()

        for col in ordered_columns:
            temp[col] = pd.to_numeric(
                temp[col],
                errors="coerce"
            )

        first_period = temp[
            ordered_columns[0]
        ]

        last_period = temp[
            ordered_columns[-1]
        ]

        trend = last_period - first_period

        result = result.merge(
            trend.groupby(
                temp["dni_hash"]
            ).mean().rename(
                f"{family}_trend"
            ),
            on="dni_hash",
            how="left"
        )

    return result


trend_2018 = create_trend_features(data_2018)
trend_2021 = create_trend_features(data_2021)
trend_2022 = create_trend_features(data_2022)


print("Activity trend features created.")

ACTIVITY TREND FEATURES
Activity trend features created.


# SECTION 16: Recency Features

In [ ]:
print("=" * 70)
print("RECENCY FEATURES")
print("=" * 70)


RECENCY_FAMILIES = [
    "pft_events",
    "pft_visits",
    "pft_days_logged",
    "pft_assignment_submissions",
    "pft_test_submissions",
    "pft_total_minutes",
    "resource_events",
    "n_resource_days",
    "n_wifi_days"
]


def create_recency_features(df):

    result = df[
        ["dni_hash"]
    ].drop_duplicates(
        subset=["dni_hash"]
    ).copy()

    for family in RECENCY_FAMILIES:

        columns = [
            col for col in df.columns
            if col.startswith(family + "_")
        ]

        if not columns:
            continue

        periods = []

        for col in columns:

            match = re.search(
                r"_(\d{4})_(\d{1,2})$",
                col
            )

            if match:

                periods.append(
                    (
                        int(match.group(1)),
                        int(match.group(2)),
                        col
                    )
                )

        periods = sorted(periods)

        if not periods:
            continue

        last_column = periods[-1][2]

        temp = df[
            ["dni_hash", last_column]
        ].copy()

        temp[last_column] = pd.to_numeric(
            temp[last_column],
            errors="coerce"
        )

        latest = (
            temp.groupby("dni_hash")[last_column]
            .mean()
            .rename(
                f"{family}_latest"
            )
            .reset_index()
        )

        result = result.merge(
            latest,
            on="dni_hash",
            how="left"
        )

    return result


recency_2018 = create_recency_features(data_2018)
recency_2021 = create_recency_features(data_2021)
recency_2022 = create_recency_features(data_2022)


print("Recency features created.")

RECENCY FEATURES
Recency features created.


# SECTION 17: Consistency / Inactivity Features

In [ ]:
print("=" * 70)
print("CONSISTENCY / INACTIVITY FEATURES")
print("=" * 70)


def create_consistency_features(df):

    result = df[
        ["dni_hash"]
    ].drop_duplicates(
        subset=["dni_hash"]
    ).copy()

    for family in [
        "pft_events",
        "pft_visits",
        "pft_days_logged",
        "pft_total_minutes"
    ]:

        columns = [
            col for col in df.columns
            if col.startswith(family + "_")
        ]

        if not columns:
            continue

        temp = df[
            ["dni_hash"] + columns
        ].copy()

        for col in columns:
            temp[col] = pd.to_numeric(
                temp[col],
                errors="coerce"
            )

        grouped = temp.groupby("dni_hash")

        mean_value = grouped[columns].mean().mean(axis=1)

        std_value = grouped[columns].std().mean(axis=1)

        zero_months = (
            grouped[columns]
            .apply(lambda x: (x == 0).sum().sum())
        )

        result = result.merge(
            mean_value.rename(
                f"{family}_consistency_mean"
            ),
            on="dni_hash",
            how="left"
        )

        result = result.merge(
            std_value.rename(
                f"{family}_consistency_std"
            ),
            on="dni_hash",
            how="left"
        )

        result = result.merge(
            zero_months.rename(
                f"{family}_inactive_periods"
            ),
            on="dni_hash",
            how="left"
        )

    return result


consistency_2018 = create_consistency_features(data_2018)
consistency_2021 = create_consistency_features(data_2021)
consistency_2022 = create_consistency_features(data_2022)


print("Consistency and inactivity features created.")

CONSISTENCY / INACTIVITY FEATURES
Consistency and inactivity features created.


# SECTION 18: Academic + Digital Feature Integration

In [ ]:
print("=" * 70)
print("ACADEMIC + DIGITAL FEATURE INTEGRATION")
print("=" * 70)


ACADEMIC_FEATURES = [
    "anyo_ingreso",
    "tipo_ingreso",
    "preferencia_seleccion",
    "estudios_p_hash",
    "estudios_m_hash",
    "dedicacion",
    "anyo_inicio_estudios",
    "es_retitulado",
    "es_adaptado",
    "nota10_hash",
    "nota14_hash",
    "nota_asig_hash",
    "rendimiento_cuat_a",
    "rendimiento_cuat_b",
    "rendimiento_total",
    "rend_total_ultimo",
    "rend_total_penultimo",
    "rend_total_antepenultimo",
    "exento_npp",
    "curso_mas_bajo",
    "curso_mas_alto",
    "cred_mat_total",
    "cred_sup_total",
    "cred_pend_sup_tit",
    "matricula_activa",
    "impagado_curso_mat",
    "practicas",
    "actividades"
]


def create_academic_student_table(df):

    available_features = [
        col for col in ACADEMIC_FEATURES
        if col in df.columns
    ]

    academic = df[
        ["dni_hash", "abandono_hash"] +
        available_features
    ].copy()

    # One student = one record
    academic = (
        academic
        .groupby("dni_hash", as_index=False)
        .first()
    )

    return academic


academic_2018 = create_academic_student_table(data_2018)
academic_2021 = create_academic_student_table(data_2021)
academic_2022 = create_academic_student_table(data_2022)


def integrate_features(
    academic,
    temporal,
    engagement,
    trend,
    recency,
    consistency
):

    result = academic.copy()

    feature_tables = [
        temporal,
        engagement.drop(
            columns=["abandono_hash"],
            errors="ignore"
        ),
        trend,
        recency,
        consistency
    ]

    for table in feature_tables:

        result = result.merge(
            table,
            on="dni_hash",
            how="left",
            suffixes=("", "_duplicate")
        )

        duplicate_columns = [
            col for col in result.columns
            if col.endswith("_duplicate")
        ]

        if duplicate_columns:
            result.drop(
                columns=duplicate_columns,
                inplace=True
            )

    return result


features_2018 = integrate_features(
    academic_2018,
    student_temporal_2018,
    engagement_2018,
    trend_2018,
    recency_2018,
    consistency_2018
)

features_2021 = integrate_features(
    academic_2021,
    student_temporal_2021,
    engagement_2021,
    trend_2021,
    recency_2021,
    consistency_2021
)

features_2022 = integrate_features(
    academic_2022,
    student_temporal_2022,
    engagement_2022,
    trend_2022,
    recency_2022,
    consistency_2022
)


print("Integrated feature tables created.")

print(
    "2018:",
    features_2018.shape
)

print(
    "2021:",
    features_2021.shape
)

print(
    "2022:",
    features_2022.shape
)

ACADEMIC + DIGITAL FEATURE INTEGRATION
Integrated feature tables created.
2018: (20233, 103)
2021: (20264, 103)
2022: (20427, 103)


# SECTION 19: Feature-Level Missingness

In [ ]:
print("=" * 70)
print("FEATURE-LEVEL MISSINGNESS")
print("=" * 70)


def missingness_report(df, dataset_name):

    report = pd.DataFrame({
        "Dataset": dataset_name,
        "Feature": df.columns,
        "Missing_Count": df.isna().sum().values,
        "Missing_Percent": (
            df.isna().mean().values * 100
        )
    })

    report = report.sort_values(
        "Missing_Percent",
        ascending=False
    )

    return report


missing_2018 = missingness_report(
    features_2018,
    "2018"
)

missing_2021 = missingness_report(
    features_2021,
    "2021"
)

missing_2022 = missingness_report(
    features_2022,
    "2022"
)


feature_missingness = pd.concat(
    [
        missing_2018,
        missing_2021,
        missing_2022
    ],
    ignore_index=True
)


display(
    feature_missingness.head(30)
)

FEATURE-LEVEL MISSINGNESS


,Dataset,Feature,Missing_Count,Missing_Percent
0,2018,pft_visits_std,20233,100.0
1,2018,pft_visits_max,20233,100.0
2,2018,pft_visits_mean,20233,100.0
3,2018,rendimiento_cuat_b,20233,100.0
4,2018,rendimiento_total,20233,100.0
5,2018,exento_npp,20233,100.0
6,2018,rendimiento_cuat_a,20233,100.0
7,2018,pft_days_logged_trend,20233,100.0
8,2018,pft_visits_trend,20233,100.0
9,2018,pft_assignment_submissions_latest,20233,100.0


# SECTION 20: Leakage Check

In [ ]:
print("=" * 70)
print("FEATURE ENGINEERING LEAKAGE CHECK")
print("=" * 70)


LEAKAGE_KEYWORDS = [
    "abandono",
    "baja",
    "withdraw",
    "dropout",
    "drop_out"
]


EXCLUDED_COLUMNS = [
    "dni_hash",
    "baja_fecha",
    "fecha_datos"
]


def leakage_check(df, dataset_name):

    findings = []

    for column in df.columns:

        column_lower = column.lower()

        keyword_flag = any(
            keyword in column_lower
            for keyword in LEAKAGE_KEYWORDS
        )

        excluded_flag = column in EXCLUDED_COLUMNS

        if keyword_flag or excluded_flag:

            findings.append({
                "Dataset": dataset_name,
                "Feature": column,
                "Reason": (
                    "Target/leakage-related name"
                    if keyword_flag
                    else "Identifier/metadata exclusion"
                )
            })

    return pd.DataFrame(findings)


leakage_2018 = leakage_check(
    features_2018,
    "2018"
)

leakage_2021 = leakage_check(
    features_2021,
    "2021"
)

leakage_2022 = leakage_check(
    features_2022,
    "2022"
)


leakage_report = pd.concat(
    [
        leakage_2018,
        leakage_2021,
        leakage_2022
    ],
    ignore_index=True
)


display(leakage_report)

FEATURE ENGINEERING LEAKAGE CHECK


,Dataset,Feature,Reason
0,2018,dni_hash,Identifier/metadata exclusion
1,2018,abandono_hash,Target/leakage-related name
2,2021,dni_hash,Identifier/metadata exclusion
3,2021,abandono_hash,Target/leakage-related name
4,2022,dni_hash,Identifier/metadata exclusion
5,2022,abandono_hash,Target/leakage-related name


In [ ]:
# REMOVE LEAKAGE / IDENTIFIERS
NON_PREDICTIVE_COLUMNS = [
    "dni_hash",
    "baja_fecha",
    "fecha_datos"
]


def remove_non_predictive_columns(df):

    return df.drop(
        columns=[
            col
            for col in NON_PREDICTIVE_COLUMNS
            if col in df.columns
        ],
        errors="ignore"
    )


model_features_2018 = remove_non_predictive_columns(
    features_2018
)

model_features_2021 = remove_non_predictive_columns(
    features_2021
)

model_features_2022 = remove_non_predictive_columns(
    features_2022
)


print("Non-predictive identifier/leakage-sensitive columns removed.")

Non-predictive identifier/leakage-sensitive columns removed.


# SECTION 21: FINAL STUDENT-LEVEL DATASET

In [ ]:
print("=" * 70)
print("FINAL STUDENT-LEVEL DATASET")
print("=" * 70)


final_features_2018 = model_features_2018.copy()
final_features_2021 = model_features_2021.copy()
final_features_2022 = model_features_2022.copy()


print("Final dataset shapes:")

print(
    f"2018 → {final_features_2018.shape}"
)

print(
    f"2021 → {final_features_2021.shape}"
)

print(
    f"2022 → {final_features_2022.shape}"
)


print("\nTarget distribution:")

for name, df in [
    ("2018", final_features_2018),
    ("2021", final_features_2021),
    ("2022", final_features_2022)
]:

    print(f"\n{name}")

    print(
        df["abandono_hash"]
        .value_counts(dropna=False)
    )

FINAL STUDENT-LEVEL DATASET
Final dataset shapes:
2018 → (20233, 102)
2021 → (20264, 102)
2022 → (20427, 102)

Target distribution:

2018
abandono_hash
B    18797
A     1436
Name: count, dtype: int64

2021
abandono_hash
B    18900
A     1364
Name: count, dtype: int64

2022
abandono_hash
B    18933
A     1494
Name: count, dtype: int64


# SECTION 22: FINAL FEATURE DATASET VALIDATION

In [ ]:
print("=" * 70)
print("FINAL FEATURE DATASET VALIDATION")
print("=" * 70)


def validate_final_dataset(df, dataset_name):

    print(f"\n{'-' * 60}")
    print(f"DATASET: {dataset_name}")
    print(f"{'-' * 60}")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print(
        "Duplicate rows:",
        df.duplicated().sum()
    )

    print(
        "Target exists:",
        "abandono_hash" in df.columns
    )

    if "abandono_hash" in df.columns:

        print(
            "Target missing:",
            df["abandono_hash"].isna().sum()
        )

    leakage_columns = [
        col for col in df.columns
        if any(
            keyword in col.lower()
            for keyword in LEAKAGE_KEYWORDS
        )
    ]

    print(
        "Potential leakage columns:",
        leakage_columns
    )


validate_final_dataset(
    final_features_2018,
    "2018"
)

validate_final_dataset(
    final_features_2021,
    "2021"
)

validate_final_dataset(
    final_features_2022,
    "2022"
)

FINAL FEATURE DATASET VALIDATION

------------------------------------------------------------
DATASET: 2018
------------------------------------------------------------
Rows: 20233
Columns: 102
Duplicate rows: 45
Target exists: True
Target missing: 0
Potential leakage columns: ['abandono_hash']

------------------------------------------------------------
DATASET: 2021
------------------------------------------------------------
Rows: 20264
Columns: 102
Duplicate rows: 2
Target exists: True
Target missing: 0
Potential leakage columns: ['abandono_hash']

------------------------------------------------------------
DATASET: 2022
------------------------------------------------------------
Rows: 20427
Columns: 102
Duplicate rows: 2
Target exists: True
Target missing: 0
Potential leakage columns: ['abandono_hash']


# SECTION 23: FEATURE DICTIONARY

In [ ]:
print("=" * 70)
print("FEATURE DICTIONARY")
print("=" * 70)


FEATURE_DESCRIPTIONS = {

    "dni_hash":
        "Anonymized student identifier used only for longitudinal linkage.",

    "abandono_hash":
        "Binary student dropout/withdrawal target variable.",

    "pft_events_overall":
        "Total platform events observed across the temporal window.",

    "pft_visits_overall":
        "Total platform visits across the temporal window.",

    "pft_days_logged_overall":
        "Total number of days with recorded platform activity.",

    "pft_assignment_submissions_overall":
        "Total assignment submissions across the observation window.",

    "pft_test_submissions_overall":
        "Total test submissions across the observation window.",

    "pft_total_minutes_overall":
        "Total recorded platform usage time.",

    "resource_events_overall":
        "Total recorded learning-resource events.",

    "n_resource_days_overall":
        "Total number of days with resource-related activity.",

    "n_wifi_days_overall":
        "Total number of days with recorded Wi-Fi activity."
}


feature_dictionary_rows = []

for column in final_features_2018.columns:

    description = FEATURE_DESCRIPTIONS.get(
        column,
        "Engineered or original feature; interpretation documented from source feature family."
    )

    feature_dictionary_rows.append({
        "Feature": column,
        "Data_Type": str(
            final_features_2018[column].dtype
        ),
        "Description": description
    })


feature_dictionary = pd.DataFrame(
    feature_dictionary_rows
)


display(
    feature_dictionary.head(30)
)

FEATURE DICTIONARY


,Feature,Data_Type,Description
0,abandono_hash,object,Binary student dropout/withdrawal target varia...
1,anyo_ingreso,float64,Engineered or original feature; interpretation...
2,tipo_ingreso,object,Engineered or original feature; interpretation...
3,preferencia_seleccion,float64,Engineered or original feature; interpretation...
4,estudios_p_hash,object,Engineered or original feature; interpretation...
5,estudios_m_hash,object,Engineered or original feature; interpretation...
6,dedicacion,object,Engineered or original feature; interpretation...
7,anyo_inicio_estudios,int64,Engineered or original feature; interpretation...
8,es_retitulado,float64,Engineered or original feature; interpretation...
9,es_adaptado,float64,Engineered or original feature; interpretation...


# SECTION 24: Save Final Feature Datasets

In [ ]:
print("=" * 70)
print("SAVING FINAL FEATURE DATASETS")
print("=" * 70)


FEATURE_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


final_features_2018.to_pickle(
    FEATURE_RESULT_DIR /
    "upv_2018_student_features.pkl"
)

final_features_2021.to_pickle(
    FEATURE_RESULT_DIR /
    "upv_2021_student_features.pkl"
)

final_features_2022.to_pickle(
    FEATURE_RESULT_DIR /
    "upv_2022_student_features.pkl"
)


feature_dictionary.to_csv(
    FEATURE_RESULT_DIR /
    "feature_dictionary.csv",
    index=False
)


feature_missingness.to_csv(
    FEATURE_RESULT_DIR /
    "feature_missingness.csv",
    index=False
)


leakage_report.to_csv(
    FEATURE_RESULT_DIR /
    "feature_engineering_leakage_check.csv",
    index=False
)


print("Final feature datasets saved successfully.")

print(
    "\nLocation:",
    FEATURE_RESULT_DIR
)

SAVING FINAL FEATURE DATASETS
Final feature datasets saved successfully.

Location: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Results/UPV/Feature_Engineering


# SECTION 25: FINAL FEATURE ENGINEERING STATUS

In [ ]:
print("=" * 70)
print("UPV FEATURE ENGINEERING STATUS")
print("=" * 70)

print("""
Completed:

1. Preprocessing outputs loaded
2. Intermediate datasets verified
3. Temporal windows identified
4. Temporal columns mapped to feature families
5. Prediction unit maintained at student level
6. Temporal features aggregated
7. Digital engagement features created
8. Activity trend features created
9. Recency features created
10. Consistency/inactivity features created
11. Academic and digital features integrated
12. Feature-level missingness profiled
13. Feature engineering leakage check performed
14. Identifier/leakage-sensitive variables excluded
15. Final student-level feature tables created
16. Feature dictionary generated
17. Final feature datasets saved

Not yet performed:

1. Train/validation/test splitting
2. Training-only imputation
3. Categorical encoding
4. Numerical scaling
5. Class imbalance strategy
6. Model training
7. Model comparison
8. Hyperparameter tuning
9. Final evaluation
10. Explainability analysis
""")

print(
    "\nFeature Engineering completed successfully."
)

UPV FEATURE ENGINEERING STATUS

Completed:

1. Preprocessing outputs loaded
2. Intermediate datasets verified
3. Temporal windows identified
4. Temporal columns mapped to feature families
5. Prediction unit maintained at student level
6. Temporal features aggregated
7. Digital engagement features created
8. Activity trend features created
9. Recency features created
10. Consistency/inactivity features created
11. Academic and digital features integrated
12. Feature-level missingness profiled
13. Feature engineering leakage check performed
14. Identifier/leakage-sensitive variables excluded
15. Final student-level feature tables created
16. Feature dictionary generated
17. Final feature datasets saved

Not yet performed:

1. Train/validation/test splitting
2. Training-only imputation
3. Categorical encoding
4. Numerical scaling
5. Class imbalance strategy
6. Model training
7. Model comparison
8. Hyperparameter tuning
9. Final evaluation
10. Explainability analysis


Feature Engineering 